In [2]:
import torch
import torch.nn.functional as F 
%matplotlib inline

In [9]:
words = open('names.txt','r').read().splitlines()
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [15]:
# vocabulary of charaters
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)} #map char:integer form 1
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [19]:
#building the dataset of the model
# the model is learning based on a context, the last n character ecountered before, in this way whe choose a context lenght size
block_size = 3 #context_lenght
#we build the dataset from our data: the first block_size character followed by the target character, for each word,
# example for the word emma we want
#  X  ----> Y
# ... ----> e
# ..e ----> m
# .em ----> m
# emm ----> a
# mma ----> .
# the value must be the integer as the model can only be trained on numbers
# the dot is used as padding
X , Y = [], []
for w in words[:5]:
    # print(w)
    context = [0]*block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        # print(''.join(itos[c] for c in context), '---->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [20]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [ ]:
# the approces used in the paper is based on word-level where they embedded the context of word
# in to ac space to small dimension
# weare modelling a charchter level - so we have an input based on the vocabolary of the 27 chars e we choose an embedde dim of 2
C = torch.randn(27,2)
# we use the pytorch indexing. we want the embedding on the input, where each input isa a block of char so we che use
emb = C[X] # in this way we get for each colom of X that was a char the correspond embedding of C
# es. 
# print(X[13,2])
# C[X[13,2]]
emb.shape

torch.Size([32, 3, 2])

In [44]:
# for the hidden layer we need choose a dimension that we want to learn, fro example 100
# for each of the input that isa a 3*2 the embedding for each context_lenght
W1 = torch.randn(6, 100)
b1 = torch.randn(100)
# but to apply mat mul of emb @ W1 wee need to take the dim of emb to 6 because they are
# ([32, 3, 2]) * [(6, 100)]
# to this we can concat all the embedding to the 1 dimension (the dimension strat from 1)
# torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]], 1).shape
# torch.Size([32, 6])
# this kind of index doesn't scale well if we want to change the context lenght, so intesad we canuse unbind
# torch.unbind delete on their own a specific dimension to give you a list/tensor of element without that
# torch.cat(torch.unbind(emb,1), 1).shape
# there is a simple and best way that is with view
# emb.view(32, 6)
# h = emb.view(32, 6) @ W1 + b1
# h.shape
# but we don't want to specify the size of 32 because it can change so we can use
# emb.view(emb.shape[0], 6)
# or a better way that is emb.view(-1, 6), -1 tall to tensor to infer that alone to mantain the shape o 6
h = torch.tanh(emb.view(32, 6) @ W1 + b1) 
h.shape

torch.Size([32, 100])

In [ ]:
# the second layer - the input dim in the otput dim of the first layer, the output dim the number of vocabolary in which we apply softmax fro porbability distribution
W2 = torch.randn(100,27)
b2 = torch.randn(27)

In [47]:
logits = h @ W2 + b2

In [48]:
logits.shape

torch.Size([32, 27])

In [53]:
# softmax
counts = logits.exp()
prob = counts / counts.sum(1, keepdim=True)
# sum(prob[1]) # each row is normalized
prob.shape


torch.Size([32, 27])

In [ ]:
# we want the probabilty for the next charcter that defined by Y
# for each input of prob we look at the character Y prob[torch.arange(32),Y]
# we use the loss fucntion of negative log likelihood
loss = -prob[torch.arange(32),Y].log().mean()
# in a simplier command, in the case of classificarion task
F.cross_entropy(logits,Y)

In [58]:
#rewriting the code
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2), generator=g)
W1 = torch.randn((6,100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100,27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [59]:
#number of pam
sum (p.nelement() for p in parameters)

3481

In [ ]:
# foward pass
emb = C[X] # (32, 3, 2)
h = torch.tanh(emb.view(-1,6) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Y)
# backward pass
# first we set the gradint to be zero
for p in parameters:
    p.grad = None # equivalent to setting to zero
loss.backward() # to populate these gradients
# update 
for p in parameters:
    p.data += -0.1 * p.grad #updating de parameters for the learning rate

In [60]:
# setting the real training just taking the code above adn duing for tot step
for p in parameters:
    p.requires_grad = True

for _ in range(10):
    # foward pass
    emb = C[X] # (32, 3, 2)
    h = torch.tanh(emb.view(-1,6) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)
    loss = F.cross_entropy(logits, Y)
    print(loss.item())
    # backward pass
    # first we set the gradint to be zero
    for p in parameters:
        p.grad = None # equivalent to setting to zero
    loss.backward() # to populate these gradients
    # update 
    for p in parameters:
        p.data += -0.1 * p.grad #updating de parameters for the learning rate

17.76971435546875
13.656400680541992
11.298768997192383
9.4524564743042
7.984262466430664
6.891321182250977
6.100014686584473
5.452036380767822
4.898151874542236
4.414664268493652
